# Notebook 3: Prompt Engineering Techniques
### From zero-shot to chain-of-thought — hands-on with a mock LLM

**No external models needed.** Uses a deterministic MockLLM that simulates realistic responses.
When your org allows API access, swap `MockLLM` for real API calls — all other code stays the same.

---
Topics covered:
1. Prompt anatomy (System / User / Assistant roles)
2. Zero-shot prompting
3. Few-shot prompting
4. Chain-of-thought (CoT)
5. Role prompting
6. Output formatting (JSON schema, XML tags)
7. Temperature and sampling parameters
8. Token cost calculator for prompt designs

In [ ]:
import json
import re
import random
from textwrap import dedent

# ============================================================
# MockLLM: A deterministic simulator for demo purposes
# Swap this class with real API calls when access is available
# ============================================================

class MockLLM:
    """
    Simulates LLM behavior for training demos.
    Shows realistic outputs for banking prompt engineering examples.
    
    To use with a real API:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            temperature=temperature
        )
        return response.choices[0].message.content
    """
    
    def __init__(self, temperature=0.1):
        self.temperature = temperature
        self.call_count = 0
    
    def chat(self, system, user, mock_response=None):
        """Simulates a chat completion call"""
        self.call_count += 1
        # Estimate tokens
        input_tokens = len((system + user).split()) * 1.3
        output_tokens = len((mock_response or "").split()) * 1.3 if mock_response else 100
        print(f"[MockLLM] Call #{self.call_count} | ~{int(input_tokens)} input tokens | temp={self.temperature}")
        print(f"{'='*60}")
        print(f"SYSTEM: {system[:200]}{'...' if len(system)>200 else ''}")
        print(f"USER: {user[:300]}{'...' if len(user)>300 else ''}")
        print(f"{'='*60}")
        if mock_response:
            print(f"RESPONSE:\n{mock_response}")
        print()
        return mock_response

llm = MockLLM(temperature=0.1)
print("MockLLM initialized ✓")
print("Replace MockLLM.chat() with real API calls when available.")

## 1. Anatomy of a Production Prompt
Three roles: **System** (operator), **User** (runtime), **Assistant** (priming)

In [ ]:
# The three roles visualized
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    PRODUCTION PROMPT ANATOMY                     ║
╠══════════════════════════════════════════════════════════════════╣
║  SYSTEM ROLE  (set once, defines model identity + constraints)   ║
║  ─────────────────────────────────────────────────────────────   ║
║  "You are a banking document analyst at an RBI-regulated bank.   ║
║  You must only answer questions based on the provided document.  ║
║  Never infer or speculate. Always respond in JSON."              ║
╠══════════════════════════════════════════════════════════════════╣
║  USER ROLE  (dynamic, injected at runtime per request)           ║
║  ─────────────────────────────────────────────────────────────   ║
║  <document>                                                      ║
║  [OCR text of KYC document injected here]                        ║
║  </document>                                                     ║
║  Extract: name, DOB, address, PAN number.                        ║
╠══════════════════════════════════════════════════════════════════╣
║  ASSISTANT ROLE  (optional prefix to force output format)        ║
║  ─────────────────────────────────────────────────────────────   ║
║  {"result":  ← forces JSON continuation                          ║
╚══════════════════════════════════════════════════════════════════╝
""")

# Practical: show API payload structure
api_payload = {
    "model": "gpt-4",   # or claude-sonnet-4-6
    "messages": [
        {"role": "system", "content": "You are a banking document analyst..."},
        {"role": "user",   "content": "<document>...</document> Extract: name, DOB, PAN."},
        # Optional: pre-fill assistant
        # {"role": "assistant", "content": '{"result":'},
    ],
    "temperature": 0.0,
    "max_tokens": 512
}
print("API Payload structure:")
print(json.dumps(api_payload, indent=2))

## 2. Zero-Shot Prompting

In [ ]:
# Zero-shot: no examples, rely on model's pre-trained capability
system_zero_shot = dedent("""
    You are a credit analyst at an Indian bank.
    Classify the following customer complaint as ONE of:
    [Loan, Account, Card, Fraud, Other]
    Respond with only the category label. Nothing else.
""").strip()

complaints = [
    ("I was charged twice for my EMI this month and my CIBIL score has dropped.",
     "Loan"),
    ("My debit card was declined at the ATM even though I have sufficient balance.",
     "Card"),
    ("There are three transactions I did not make on my account totalling Rs. 45,000.",
     "Fraud"),
    ("I need to update my mobile number linked to my savings account.",
     "Account"),
    ("The bank branch near my house has very slow service.",
     "Other"),
]

print("=== ZERO-SHOT COMPLAINT CLASSIFICATION ===")
print(f"{'Complaint (truncated)':<60} {'Expected':>10} {'Model':>10} {'✓?':>5}")
print("-" * 92)
correct = 0
for complaint, expected in complaints:
    # In real deployment: response = llm.chat(system_zero_shot, complaint)
    # Here we simulate the correct model response
    response = expected  # Simulated perfect model
    match = "✓" if response == expected else "✗"
    if response == expected: correct += 1
    print(f"{complaint[:58]:<60} {expected:>10} {response:>10} {match:>5}")

print(f"\nAccuracy: {correct}/{len(complaints)} = {correct/len(complaints):.0%}")
print("\nKey point: Zero-shot works well when categories are clear and the model")
print("has seen similar tasks in pre-training. Fails on bank-specific edge cases.")

## 3. Few-Shot Prompting — Adding Examples

In [ ]:
# Few-shot: provide examples to teach format and boundary cases

def build_few_shot_prompt(examples, query):
    """Build a few-shot prompt from (input, output) pairs"""
    prompt = "Classify the customer complaint into: Loan, Account, Card, Fraud, Other.\n\n"
    for ex_input, ex_output in examples:
        prompt += f"Complaint: {ex_input}\nCategory: {ex_output}\n\n"
    prompt += f"Complaint: {query}\nCategory:"
    return prompt

# Carefully selected examples — cover boundary cases
examples = [
    ("My home loan EMI was deducted twice in the same month.", "Loan"),
    ("I received an OTP I didn't request — someone may be trying to access my account.", "Fraud"),
    ("My credit card reward points are not showing correctly.", "Card"),
    ("I want to add a nominee to my fixed deposit.", "Account"),
    # Edge case: could be Loan or Fraud — example shows it's Fraud
    ("An EMI I never applied for has started appearing in my account statement.", "Fraud"),
]

# Edge case query — ambiguous between Loan and Fraud
ambiguous_query = "A personal loan disbursement appeared in my account but I never applied for it."

prompt = build_few_shot_prompt(examples, ambiguous_query)
print("=== FEW-SHOT PROMPT ===")
print(prompt)
print("=" * 60)
print("\nExpected output with few-shot: Fraud")
print("Without the fraud example, zero-shot might say: Loan")
print("\nKey insight: 3 well-chosen examples > 10 mediocre ones.")

In [ ]:
# Token cost comparison: zero-shot vs few-shot
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    count = lambda t: len(enc.encode(t))
except:
    count = lambda t: len(t.split()) * 1.3

zero_shot_system = "Classify the customer complaint into: Loan, Account, Card, Fraud, Other."
zero_shot_tokens = count(zero_shot_system + ambiguous_query)
few_shot_tokens = count(prompt)

calls_per_day = 50_000
cost_per_m = 5.0  # Claude Opus input

zero_shot_daily = (zero_shot_tokens * calls_per_day / 1e6) * cost_per_m
few_shot_daily = (few_shot_tokens * calls_per_day / 1e6) * cost_per_m

print(f"Zero-shot prompt tokens : {int(zero_shot_tokens):>6}")
print(f"Few-shot prompt tokens  : {int(few_shot_tokens):>6}  ({int(few_shot_tokens/zero_shot_tokens)}x more)")
print(f"\nAt {calls_per_day:,} calls/day:")
print(f"  Zero-shot daily cost   : ${zero_shot_daily:.2f}")
print(f"  Few-shot daily cost    : ${few_shot_daily:.2f}")
print(f"  Extra cost for examples: ${few_shot_daily - zero_shot_daily:.2f}/day")
print("\nConclusion: If few-shot reduces error rate significantly, the extra cost is worth it.")

## 4. Chain-of-Thought (CoT) — Loan Eligibility

In [ ]:
# From the deck: the worked FOIR example as runnable code

cot_system = dedent("""
    You are a credit risk analyst at an Indian bank.
    Reason through each step before concluding.
    Format your response EXACTLY as:
    
    STEP 1 - FOIR CHECK: [calculation + policy comparison]
    STEP 2 - CIBIL CHECK: [score + threshold]
    STEP 3 - STABILITY CHECK: [employment tenure]
    STEP 4 - DECISION: [Approve/Conditional Decline/Decline] + [reason]
    STEP 5 - RECOMMENDATION: [actionable next step for the customer]
""").strip()

applicant = dedent("""
    Applicant: Mr. Rajesh Kumar
    Monthly Income: ₹1,20,000
    Existing Car EMI: ₹25,000
    Existing Personal Loan EMI: ₹8,000
    Requested Loan: ₹50,00,000 @ 8.75% for 20 years
    Requested EMI (calculated): ₹43,900/month
    CIBIL Score: 712
    Employment: Salaried, 4 years in current role
    
    Bank Policy:
    - Max FOIR: 55%
    - Min CIBIL: 700
    - Min employment tenure: 2 years
""").strip()

# Simulated CoT response (as a real model would produce)
cot_response = dedent("""
    STEP 1 - FOIR CHECK:
    Total fixed obligations = ₹25,000 (car) + ₹8,000 (personal loan) + ₹43,900 (proposed EMI)
    Total obligations = ₹76,900
    FOIR = ₹76,900 / ₹1,20,000 = 64.1%
    Policy max = 55% → BREACH of 9.1% identified. ❌
    
    STEP 2 - CIBIL CHECK:
    CIBIL score = 712
    Policy minimum = 700
    712 > 700 → ACCEPTABLE ✓
    
    STEP 3 - STABILITY CHECK:
    Current employment tenure = 4 years
    Policy minimum = 2 years
    4 > 2 → ACCEPTABLE ✓
    
    STEP 4 - DECISION: Conditional Decline
    Primary disqualifier: FOIR of 64.1% exceeds policy maximum of 55% by 9.1%.
    CIBIL and employment criteria are both satisfied.
    Confidence: High. All figures applied deterministically against stated policy.
    
    STEP 5 - RECOMMENDATION:
    Maximum eligible loan at 55% FOIR:
    Available EMI headroom = (55% × ₹1,20,000) - ₹25,000 - ₹8,000 = ₹66,000 - ₹33,000 = ₹33,000
    At ₹33,000 EMI, eligible loan ≈ ₹37,50,000.
    Suggest: Apply for ₹38L. Alternatively, foreclose personal loan first to free ₹8,000 EMI headroom,
    then reapply for full ₹50L.
""").strip()

print("=== CHAIN-OF-THOUGHT PROMPT ===")
print(f"SYSTEM: {cot_system}\n")
print(f"USER:\n{applicant}\n")
print("=" * 60)
print("RESPONSE (simulated):")
print(cot_response)

In [ ]:
# Extract structured decision from CoT output (post-processing)
def parse_cot_decision(cot_output):
    """Extract decision fields from CoT response"""
    result = {}
    
    # Extract FOIR
    foir_match = re.search(r'FOIR\s*=\s*[₹\d,/\s]+\s*=\s*(\d+\.\d+)%', cot_output)
    if foir_match:
        result['foir_pct'] = float(foir_match.group(1))
    
    # Extract decision
    dec_match = re.search(r'DECISION:\s*(Approve|Conditional Decline|Decline)', cot_output)
    if dec_match:
        result['decision'] = dec_match.group(1)
    
    # Check CIBIL pass/fail
    result['cibil_pass'] = 'ACCEPTABLE ✓' in cot_output
    result['stability_pass'] = 'ACCEPTABLE ✓' in cot_output
    
    return result

parsed = parse_cot_decision(cot_response)
print("Parsed structured output from CoT reasoning:")
print(json.dumps(parsed, indent=2))

print("\n=== Why CoT for Regulated Decisions? ===")
print("1. Full audit trail — every number is traceable")
print("2. Regulator can verify the reasoning, not just the outcome")
print("3. 'The model said so' is NOT an audit trail — the CoT is")
print("4. Improves accuracy on multi-step calculations vs direct answer")

## 5. Role Prompting

In [ ]:
# Three different personas for the same underlying task
# Watch how the persona changes the output style

personas = {
    "Generic Assistant": """
        You are a helpful assistant. Answer questions about banking.
    """,
    
    "Credit Underwriter": """
        You are a credit underwriter at a PSU bank following RBI prudential norms.
        When summarising bureau reports, highlight: repayment history, utilisation ratio,
        derogatory marks, and recent enquiries.
        You never infer intent — you report facts only.
        Format output as: SUMMARY | KEY_FLAGS | RECOMMENDATION
    """,
    
    "Compliance Analyst": """
        You are a senior compliance analyst at an RBI-regulated scheduled commercial bank
        with 15 years of experience in AML and KYC procedures.
        You are precise, cite specific regulations where applicable,
        and flag ambiguities without making assumptions.
        Severity levels: CRITICAL | HIGH | MEDIUM | LOW
    """,
    
    "Code Reviewer": """
        You are a senior Java engineer specialising in core banking systems on Temenos T24.
        Review code for: correctness, thread safety, transaction isolation, OWASP Top 10.
        Flag issues with severity: CRITICAL | HIGH | MEDIUM | LOW.
        Never suggest refactors — only flag actual defects and security issues.
    """
}

# Same input, different personas
input_text = "Customer has 3 active loans, CIBIL 680, missed 2 payments in last 12 months."

# Simulated responses per persona
responses = {
    "Generic Assistant": 
        "This customer has 3 loans and a low CIBIL score with some missed payments, which could indicate financial stress.",
    
    "Credit Underwriter":
        """SUMMARY: Applicant carries 3 active credit facilities. CIBIL score of 680 is below the 700 threshold for retail lending.
        Repayment history shows 2 delinquencies (DPD30+) in the past 12 months.
        KEY_FLAGS: [1] Sub-threshold CIBIL (680 < 700) [2] Recent delinquency pattern — 2 in 12M [3] High credit facility count
        RECOMMENDATION: DECLINE at pre-screening. Refer for Recoveries review if existing customer.""",
    
    "Compliance Analyst":
        """RISK ASSESSMENT — AML/KYC Perspective:
        HIGH: Multiple active credit facilities (3) with deteriorating repayment pattern may indicate debt stress.
        Under RBI Master Circular on KYC (RBI/2023-24/26), periodic KYC refresh is mandatory for high-risk customers.
        MEDIUM: CIBIL decline trajectory should be flagged for Enhanced Due Diligence (EDD) if customer has trade relationships.
        Ambiguity: Nature of 3 loans not specified — if any are from NBFCs, cross-reporting gap may exist.""",
    
    "Code Reviewer":
        "Input is a business scenario, not code. Cannot perform code review. Please provide Java source code for review."
}

for persona, system in personas.items():
    print(f"\n{'='*60}")
    print(f"PERSONA: {persona}")
    print(f"SYSTEM PROMPT: {system.strip()[:120]}...")
    print(f"USER: {input_text}")
    print(f"RESPONSE:")
    print(responses[persona])

## 6. Output Formatting — JSON Schema Enforcement

In [ ]:
# JSON schema in system prompt + XML tag trick

json_schema_system = dedent("""
    You are a KYC extraction engine.
    Extract fields from the document provided.
    
    Respond ONLY with this JSON structure. No other text:
    {
      "name": "string or null",
      "date_of_birth": "DD/MM/YYYY or null",
      "pan_number": "string matching [A-Z]{5}[0-9]{4}[A-Z] or null",
      "aadhaar_last4": "4-digit string or null",
      "address": "string or null",
      "extraction_confidence": "High|Medium|Low",
      "missing_fields": ["list of field names that are null"]
    }
""").strip()

# XML tags create unambiguous boundaries in the prompt
kyc_user = dedent("""
    <document type="PAN_CARD">
    INCOME TAX DEPARTMENT
    GOVT. OF INDIA
    Permanent Account Number Card
    
    RAJESH KUMAR
    Father: SURESH KUMAR
    Date of Birth: 15/03/1985
    
    ABCDE1234F
    </document>
    
    <document type="AADHAAR">
    UNIQUE IDENTIFICATION AUTHORITY OF INDIA
    Rajesh Kumar
    DOB: 15/03/1985
    Male
    XXXX XXXX 5678
    42, MG Road, Bangalore - 560001
    </document>
""").strip()

# Simulated model output
simulated_json_output = json.dumps({
    "name": "RAJESH KUMAR",
    "date_of_birth": "15/03/1985",
    "pan_number": "ABCDE1234F",
    "aadhaar_last4": "5678",
    "address": "42, MG Road, Bangalore - 560001",
    "extraction_confidence": "High",
    "missing_fields": []
}, indent=2)

print("SYSTEM PROMPT:")
print(json_schema_system)
print("\nUSER (with XML tags):")
print(kyc_user)
print("\n" + "="*60)
print("RESPONSE (machine-parseable JSON):")
print(simulated_json_output)

# Validate the JSON
try:
    parsed = json.loads(simulated_json_output)
    pan_valid = bool(re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', parsed['pan_number'] or ''))
    dob_valid = bool(re.match(r'^\d{2}/\d{2}/\d{4}$', parsed['date_of_birth'] or ''))
    print(f"\nPost-processing validation:")
    print(f"  PAN format valid: {pan_valid}")
    print(f"  DOB format valid: {dob_valid}")
    print(f"  Missing fields  : {parsed['missing_fields']}")
    print(f"  Ready for downstream processing: {'YES' if pan_valid and dob_valid else 'NO'}")
except json.JSONDecodeError as e:
    print(f"JSON parse failed: {e} — add retry logic in production")

## 7. Temperature — Visualizing Randomness

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def softmax_with_temperature(logits, temperature):
    """The actual formula: softmax(logits / temperature)"""
    if temperature == 0:
        # Greedy: all probability mass on argmax
        probs = np.zeros_like(logits, dtype=float)
        probs[np.argmax(logits)] = 1.0
        return probs
    scaled = np.array(logits) / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))  # numerical stability
    return exp_scaled / exp_scaled.sum()

# Simulate next-token probabilities after "The loan application was..."
# Raw logits from the model (unnormalized)
next_tokens = ["approved", "declined", "reviewed", "processed", "submitted", "cancelled"]
raw_logits  = [4.2,       2.8,       2.1,       1.8,        1.2,          0.9]

temperatures = [0.0, 0.2, 0.7, 1.0, 1.5]

fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 5))
colors = ['steelblue', 'mediumseagreen', 'darkorange', 'tomato', 'purple']

for ax, temp, color in zip(axes, temperatures, colors):
    probs = softmax_with_temperature(raw_logits, temp)
    bars = ax.bar(next_tokens, probs, color=color, alpha=0.8, edgecolor='black')
    ax.set_title(f'Temperature = {temp}\n{"(Greedy)" if temp == 0 else "(Creative)" if temp > 1.2 else ""}', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(next_tokens, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Probability' if ax == axes[0] else '')
    for bar, prob in zip(bars, probs):
        if prob > 0.02:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{prob:.2f}', ha='center', fontsize=7)

plt.suptitle('Effect of Temperature on Token Selection\n"The loan application was..."', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/temperature_effect.png', dpi=150, bbox_inches='tight')
plt.close()
print("Temperature visualization saved.")

print("\n=== Banking Temperature Guide ===")
print(f"{'Task':<45} {'Temp':>6} {'Why'}")
print("-" * 85)
guidelines = [
    ("KYC field extraction (JSON)",              0.0, "Deterministic required"),
    ("Loan decision classification",              0.0, "Same input must = same output"),
    ("Bureau report summarisation",               0.1, "Factual, minimal creativity"),
    ("Customer complaint routing",                0.1, "Structured output"),
    ("Policy Q&A (RAG)",                          0.2, "Grounded in document"),
    ("Internal newsletter draft",                 0.7, "Some creativity acceptable"),
    ("Marketing tagline generation",              1.0, "Creative diversity needed"),
    ("Random hallucination territory",            1.5, "NOT for banking tasks"),
]
for task, temp, reason in guidelines:
    flag = "⚠️ " if temp >= 1.0 else "✓  "
    print(f"{flag} {task:<42} {temp:>6.1f}   {reason}")

## 8. Prompt Drift — Why Prompts Need Version Control

In [ ]:
# Demonstrate prompt as versioned artifact

class PromptRegistry:
    """A minimal prompt version registry — real systems use Git or a prompt management tool"""
    
    def __init__(self):
        self.registry = {}
    
    def register(self, name, version, system_prompt, change_note):
        key = f"{name}@{version}"
        self.registry[key] = {
            "name": name,
            "version": version,
            "system_prompt": system_prompt,
            "change_note": change_note,
            "char_count": len(system_prompt),
        }
        return key
    
    def get(self, name, version):
        return self.registry.get(f"{name}@{version}")
    
    def list_versions(self, name):
        return [v for k, v in self.registry.items() if v['name'] == name]

registry = PromptRegistry()

registry.register(
    "complaint_classifier", "v1.0",
    "Classify the complaint as: Loan, Account, Card, Fraud, Other.",
    "Initial version"
)
registry.register(
    "complaint_classifier", "v1.1",
    "Classify the customer complaint as ONE of: Loan, Account, Card, Fraud, Other. Respond with only the label.",
    "Added 'ONE of' constraint after model started returning explanations"
)
registry.register(
    "complaint_classifier", "v1.2",
    "You are a banking operations classifier. Classify the customer complaint as ONE of: [Loan, Account, Card, Fraud, Other]. Output: exactly one label from the list, nothing else.",
    "Added role + explicit output constraint after GPT-4 model update broke v1.1"
)

print("=== PROMPT VERSION HISTORY ===")
for v in registry.list_versions("complaint_classifier"):
    print(f"\nVersion {v['version']} ({v['char_count']} chars)")
    print(f"  Change: {v['change_note']}")
    print(f"  Prompt: {v['system_prompt']}")

print("\n=== Production Checklist for Prompts ===")
checklist = [
    "Store prompts in Git, not in config files or Confluence",
    "Every prompt change = new version = new eval run",
    "Run eval on every model version bump (model updates can break prompts silently)",
    "Log prompt version with every API response in production",
    "A/B test prompt versions before full rollout",
]
for item in checklist:
    print(f"  ☐ {item}")

## Summary

| Technique | When to Use | Banking Example |
|-----------|------------|----------------|
| Zero-shot | Simple, well-defined tasks | Complaint category classification |
| Few-shot | Edge cases, domain-specific formats | Ambiguous complaint routing |
| Chain-of-Thought | Multi-step reasoning, audit trails | Loan eligibility, FOIR calculation |
| Role prompting | Consistent output style/persona | Credit underwriter, compliance analyst |
| JSON schema | Machine-parseable output | KYC extraction, structured decisions |
| Temperature 0.0 | Deterministic, compliance tasks | Any regulated output |
| Version control | All production prompts | Every prompt is versioned in Git |